In [8]:
import numpy as np
import pandas as pd

In [9]:
from pathlib import Path

In [13]:
data_dir = Path(
    "/Users/sahil/CircuitCost/CircuitCost/ML4Accel-Dataset/"
    "fpga_ml_dataset/HLS_dataset/polybench/"
    "gemm/xczu9eg-ffvb1156-2-i"
)

hls = pd.read_csv(data_dir / "post_hls_info.csv")
impl = pd.read_csv(data_dir / "post_implementation_info.csv")
hls.head()

,prj,clk_target(ns),clk_estimated(ns),clk_uncertainty(ns),bram,dsp,ff,lut,c_num_arith,c_num_logic,rtl_num_arith,rtl_num_logic,data_input_port(numxbits),data_output_port(numxbits)
0,io1_l2n1n1_l4n1n1,10.0,8.419,1.25,32,5,758,1239,22,9,19,9,5x32,1x32
1,io1_l2n1n1_l4n1p1,10.0,8.419,1.25,32,5,933,1356,23,9,20,9,5x32,1x32
2,io1_l2n1n1_l4p1n1,10.0,9.298,1.25,32,5,4578,3914,146,71,17,71,5x32,1x32
3,io1_l2n1p1_l4n1n1,10.0,8.419,1.25,32,5,845,1416,24,12,21,12,5x32,1x32
4,io1_l2n1p1_l4n1p1,10.0,8.419,1.25,32,5,1020,1519,25,12,22,12,5x32,1x32


In [15]:
#Preparing features from HLS table
feature_columns = [
    "clk_target(ns)",
    "clk_estimated(ns)",
    "clk_uncertainty(ns)",
    "bram",
    "dsp",
    "ff",
    "lut",
    "c_num_arith",
    "c_num_logic",
    "rtl_num_arith",
    "rtl_num_logic"
]

In [17]:
#Defining X and y
X = hls[feature_columns].copy()
y = impl["Total LUTs"].copy()
X.head()

,clk_target(ns),clk_estimated(ns),clk_uncertainty(ns),bram,dsp,ff,lut,c_num_arith,c_num_logic,rtl_num_arith,rtl_num_logic
0,10.0,8.419,1.25,32,5,758,1239,22,9,19,9
1,10.0,8.419,1.25,32,5,933,1356,23,9,20,9
2,10.0,9.298,1.25,32,5,4578,3914,146,71,17,71
3,10.0,8.419,1.25,32,5,845,1416,24,12,21,12
4,10.0,8.419,1.25,32,5,1020,1519,25,12,22,12


In [18]:
y.head()

0     645
1     656
2    4131
3     813
4     833
Name: Total LUTs, dtype: int64

In [19]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 82.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 107.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [23]:
#Splitting the Dataset into Training,Validation and Testing
from sklearn.model_selection import train_test_split

#First Split : 70% Training, 30% remaining
X_train, X_remaining, y_train, y_remaining = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)


In [24]:
#Second split: divide the remaining 30% equally
X_val, X_test, y_val, y_test = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.50,
    random_state=42
)

In [25]:
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (371, 11) (371,)
Validation: (80, 11) (80,)
Testing: (80, 11) (80,)


In [26]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [27]:
print("Training means:", X_train_scaled.mean(axis=0))
print("Training standard deviations:", X_train_scaled.std(axis=0))

Training means: [ 0.00000000e+00  5.43919507e-15  0.00000000e+00 -9.81544884e-17
 -1.19700596e-18  4.78802383e-18  3.35161668e-17  2.39401191e-18
 -7.18203574e-17 -1.04139518e-16 -7.18203574e-17]
Training standard deviations: [0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1.]
